In [4]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

In [6]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Βέλτιστος συνδυασμός από Φάση 1
texts_train = (train['title'].fillna('') + ' ' + train['text'].fillna('').str[:550]).tolist()
texts_valid = (valid['title'].fillna('') + ' ' + valid['text'].fillna('').str[:550]).tolist()
texts_test  = (test['title'].fillna('')  + ' ' + test['text'].fillna('').str[:550]).tolist()

texts_train = [str(t) for t in texts_train]
texts_valid = [str(t) for t in texts_valid]
texts_test  = [str(t) for t in texts_test]

y_train_hazard  = train['hazard-category']
y_train_product = train['product-category']
y_valid_hazard  = valid['hazard-category']
y_valid_product = valid['product-category']

print(f'Train: {len(texts_train)} δείγματα')
print(f'Valid: {len(texts_valid)} δείγματα')
print(f'Test:  {len(texts_test)} δείγματα')

Train: 5082 δείγματα
Valid: 565 δείγματα
Test:  997 δείγματα


In [8]:
def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product,
                       verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard,
                         average='macro', zero_division=0)
    
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)
    
    correct_hazard_mask = (y_true_hazard == y_pred_hazard)
    n_correct = correct_hazard_mask.sum()
    
    if n_correct == 0:
        f1_product = 0.0
    else:
        f1_product = f1_score(
            y_true_product[correct_hazard_mask],
            y_pred_product[correct_hazard_mask],
            average='macro', zero_division=0
        )
    
    score = (f1_hazard + f1_product) / 2
    
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {n_correct}/{len(y_true_hazard)} ({100*n_correct/len(y_true_hazard):.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    
    return score

In [9]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

PUBMED_MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'

print(f'Φόρτωση tokenizer: {PUBMED_MODEL}')
pubmed_tokenizer = AutoTokenizer.from_pretrained(PUBMED_MODEL)

print(f'Φόρτωση μοντέλου: {PUBMED_MODEL}')
pubmed_model = AutoModel.from_pretrained(PUBMED_MODEL)
pubmed_model.eval()

print('PubMedBERT φορτώθηκε!')
print(f'Παράμετροι: {sum(p.numel() for p in pubmed_model.parameters()):,}')

Φόρτωση tokenizer: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Φόρτωση μοντέλου: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PubMedBERT φορτώθηκε!
Παράμετροι: 109,482,240


## Cell PubMedBERT-1: Φόρτωση PubMedBERT

Φόρτωση του `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext`
— ένα BERT μοντέλο εκπαιδευμένο αποκλειστικά σε βιοϊατρικά
κείμενα από το PubMed.

**Γιατί PubMedBERT αντί για DistilBERT:**
Το domain μας (food hazards, bacteria, allergens, chemicals)
είναι πολύ κοντά σε βιοϊατρική ορολογία. Το PubMedBERT
γνωρίζει λέξεις όπως `listeria`, `salmonella`, `allergen`
που το γενικό DistilBERT δεν έχει δει αρκετά.

| Μοντέλο | Παράμετροι | Εκπαίδευση |
|---|---|---|
| DistilBERT | 66M | Γενικό κείμενο |
| **PubMedBERT** | **109M** | Βιοϊατρικά κείμενα ✅ |

Τα embeddings (768 διαστάσεις) θα δοθούν στο LinearSVC.

In [10]:
def get_pubmed_embeddings(texts, batch_size=16):
    """
    Παίρνει PubMedBERT embeddings για μια λίστα κειμένων.
    Ίδια λογική με το DistilBERT — παίρνουμε το [CLS] token.
    Χρησιμοποιούμε batch_size=16 (αντί 32) γιατί το μοντέλο
    είναι μεγαλύτερο και χρειάζεται περισσότερη μνήμη.
    """
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        encoded = pubmed_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )
        
        with torch.no_grad():
            outputs = pubmed_model(**encoded)
        
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_embeddings)
        
        print(f'  Επεξεργασία: {min(i+batch_size, len(texts))}/{len(texts)}', end='\r')
    
    return np.vstack(all_embeddings)


print('Εξαγωγή embeddings — Train set...')
X_train_pubmed = get_pubmed_embeddings(texts_train)
print(f'\nTrain embeddings: {X_train_pubmed.shape}')

print('Εξαγωγή embeddings — Validation set...')
X_valid_pubmed = get_pubmed_embeddings(texts_valid)
print(f'\nValid embeddings: {X_valid_pubmed.shape}')

print('Εξαγωγή embeddings — Test set...')
X_test_pubmed = get_pubmed_embeddings(texts_test)
print(f'\nTest embeddings: {X_test_pubmed.shape}')

Εξαγωγή embeddings — Train set...
  Επεξεργασία: 5082/5082
Train embeddings: (5082, 768)
Εξαγωγή embeddings — Validation set...
  Επεξεργασία: 565/565
Valid embeddings: (565, 768)
Εξαγωγή embeddings — Test set...
  Επεξεργασία: 997/997
Test embeddings: (997, 768)


In [12]:
# SVM για hazard
clf_pubmed_hazard = LinearSVC(
    C=0.5,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)
clf_pubmed_hazard.fit(X_train_pubmed, y_train_hazard)
print('PubMedBERT+SVM Hazard trained!')

# SVM για product
clf_pubmed_product = LinearSVC(
    C=0.5,
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)
clf_pubmed_product.fit(X_train_pubmed, y_train_product)
print('PubMedBERT+SVM Product trained!')

# Αξιολόγηση
pred_pubmed_hazard  = clf_pubmed_hazard.predict(X_valid_pubmed)
pred_pubmed_product = clf_pubmed_product.predict(X_valid_pubmed)

print('\n=== ΑΞΙΟΛΟΓΗΣΗ PubMedBERT+SVM ===\n')
score_pubmed = official_st1_score(
    valid['hazard-category'], pred_pubmed_hazard,
    valid['product-category'], pred_pubmed_product
)

print(f'\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'DistilBERT + SVM (feature extraction): 0.5260')
print(f'TF-IDF + SVM (tuned):                  0.7436')
print(f'PubMedBERT + SVM (feature extraction): {score_pubmed:.4f}')

PubMedBERT+SVM Hazard trained!
PubMedBERT+SVM Product trained!

=== ΑΞΙΟΛΟΓΗΣΗ PubMedBERT+SVM ===

macro-F1 Hazard:                    0.5267
Σωστά hazard predictions:           467/565 (82.7%)
macro-F1 Product (given correct h): 0.3315
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.4291

=== ΣΥΓΚΡΙΣΗ ===
DistilBERT + SVM (feature extraction): 0.5260
TF-IDF + SVM (tuned):                  0.7436
PubMedBERT + SVM (feature extraction): 0.4291


## Cell PubMedBERT: PubMedBERT + SVM (Feature Extraction)

Δοκιμή domain-specific μοντέλου εκπαιδευμένου σε βιοϊατρικά
κείμενα (PubMed) για feature extraction.

**Αποτελέσματα:**
| Μοντέλο | Score |
|---|---|
| PubMedBERT + SVM | 0.4291 |
| DistilBERT + SVM | 0.5260 |
| **TF-IDF + SVM (tuned)** | **0.7436** |

**Συμπέρασμα:** Το feature extraction χωρίς fine-tuning
αποδίδει χειρότερα από το TF-IDF ανεξάρτητα από το
domain του μοντέλου. Το TF-IDF μαθαίνει ακριβώς από
τα δικά μας δεδομένα κάτι που είναι πιο σημαντικό.